# Library

In [56]:
import requests
import feedparser
import re
from urllib.parse import quote
from html import unescape
from bs4 import BeautifulSoup
from googlenewsdecoder import new_decoderv1

## Get data

In [57]:
def get_real_article_data(google_url):
    """
    Decodes the Google URL, visits the site, and extracts a REAL
    summary and image to avoid 'Title and Summary same' issues.
    """
    try:
        # Decode Google News Link
        decoded_url = new_decoderv1(google_url, interval=1)
        if not decoded_url or not decoded_url.get('decoded_url'):
            return google_url, None, "No summary available."

        real_url = decoded_url['decoded_url']
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

        response = requests.get(real_url, headers=headers, timeout=12)
        soup = BeautifulSoup(response.text, 'html.parser')

        # 1. EXTRACT IMAGE
        img_tag = soup.find("meta", property="og:image") or soup.find("meta", attrs={"name": "twitter:image"})
        image_url = img_tag["content"] if img_tag else None

        # 2. EXTRACT REAL SUMMARY (Scrape first 2 paragraphs)
        # We ignore short text like 'Share this' or 'Follow us' by checking length
        paragraphs = soup.find_all('p')
        valid_paras = [p.get_text().strip() for p in paragraphs if len(p.get_text().strip()) > 60]

        if valid_paras:
            # Combine the first two paragraphs for a better summary
            summary = " ".join(valid_paras[:2])
            if len(summary) > 350:
                summary = summary[:350] + "..."
        else:
            summary = "Could not extract text summary from page."

        return real_url, image_url, summary

    except Exception as e:
        print(f"Scraping Error: {e}")
        return google_url, None, "Summary fetch failed."

In [58]:
def clean_title(title):
    # Removes the '- Source Name' often added by Google News
    return re.sub(r'\s-\s.*$', '', unescape(title)).strip()

## Telegram
want to change the one you send change here

In [59]:
# --- CONFIGURATION ---
TELEGRAM_BOT_TOKEN = os.getenv('TELEGRAM_TOKEN')TELEGRAM_CHAT_ID = '@wage_testing'

def send_to_telegram(title, google_link, SOURCES):
    # Fetch actual data from the source website
    real_url, image_url, real_summary = get_real_article_data(google_link)

    # Double check domain filter
    if not any(domain in real_url for domain in SOURCES):
        return

    text = (
        f"\n<b>Title:</b> {clean_title(title)}\n\n"
        f"<b>Summary:</b> {real_summary}\n\n"
        f"🔗 <a href='{real_url}'>Read Full Article</a>"
    )

    if image_url:
        api_url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendPhoto"
        payload = {"chat_id": TELEGRAM_CHAT_ID, "photo": image_url, "caption": text, "parse_mode": "HTML"}
    else:
        api_url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
        payload = {"chat_id": TELEGRAM_CHAT_ID, "text": text, "parse_mode": "HTML"}

    requests.post(api_url, data=payload)

## Query
want 1 day or all articles without day change here

In [60]:
def query(SOURCES):
    # Focus on "minimum wage" in title or the specific 2026 Decree
    site_query = " OR ".join([f"site:{s}" for s in SOURCES])
    # keyword_query = 'intitle:"minimum wage" OR intitle:"Decree 293"'
    keyword_query = 'intitle:("minimum wage" OR "national minimum wage")'

    ### for 1 day: when:1d
    full_query = quote(f"{keyword_query} ({site_query}) when:1d")

    ### If you want to check all time, just remove the time filter:
    # full_query = quote(f"{keyword_query} ({site_query})")
    
    return full_query

## Process
call and send here

In [61]:
def process(url, SOURCES):
    feed = feedparser.parse(url)

    print(f"Checking for updates...")

    processed_count = 0
    for entry in feed.entries[:10]:
        title_lower = entry.title.lower()
        # Secondary title filter for relevance
        if "minimum wage" in title_lower or "decree 293" in title_lower:
            print(f"Processing: {entry.title[:50]}...")
            send_to_telegram(entry.title, entry.link, SOURCES)
            processed_count += 1

    print(f"Task complete. Sent {processed_count} relevant articles.")

# Vietnam

In [62]:
SOURCES_viet = [
    "vietnamnews.vn", "vnexpress.net", "vietnamplus.vn",
    "english.vov.vn", "tuoitrenews.vn", "en.nhandan.vn"
]
full_query = query(SOURCES_viet)

rss_url = f"https://news.google.com/rss/search?q={full_query}&hl=vi-VN&gl=VN&ceid=VN:vi"

process(rss_url, SOURCES_viet)

Checking for updates...
Task complete. Sent 0 relevant articles.


# Lao

In [63]:
SOURCES_lao = [
    "laotiantimes.com"
]

full_query = query(SOURCES_lao)

rss_url = f"https://news.google.com/rss/search?q={full_query}&hl=la&gl=LA&ceid=LA:la"

process(rss_url, SOURCES_lao)

Checking for updates...
Task complete. Sent 0 relevant articles.


# Thailand

In [64]:
SOURCES_thai = [
    "bangkokpost.com",
    "nationthailand.com",
    "thethaiger.com",
    "thaipbsworld.com",
    "pattayamail.com"
]

full_query = query(SOURCES_thai)

rss_url = f"https://news.google.com/rss/search?q={full_query}&hl=en-TH&gl=TH&ceid=TH:en"

process(rss_url, SOURCES_thai)

Checking for updates...
Task complete. Sent 0 relevant articles.


# Myanmar

In [65]:
SOURCES_myanmar = [
    "myanmar-now.org",
    "gnlm.com.mm",
    "irrawaddy.com"
]
full_query = query(SOURCES_myanmar)

rss_url = f"https://news.google.com/rss/search?q={full_query}&hl=my-MM&gl=MM&ceid=MM:my"
process(rss_url, SOURCES_myanmar)

Checking for updates...
Task complete. Sent 0 relevant articles.


# Sri Lanka

In [66]:
SOURCES_sri_lanka = [
    "dailymirror.lk",
    "news.lk",
    "english.newsfirst.lk",
    "adaderana.lk",
    "island.lk",
    "dailynews.lk"
]

full_query = query(SOURCES_sri_lanka)

rss_url = f"https://news.google.com/rss/search?q={full_query}&hl=en-LK&gl=LK&ceid=LK:en"
process(rss_url, SOURCES_sri_lanka)

Checking for updates...
Task complete. Sent 0 relevant articles.


# Indonesia

In [67]:
SOURCES_indo = [
    "jakartaglobe.id",
    "antaranews.com",
    "thejakartapost.com",
    "aljazeera.com",
    "kompas.com"
]

full_query = query(SOURCES_indo)

rss_url = f"https://news.google.com/rss/search?q={full_query}&hl=en-ID&gl=ID&ceid=ID:en"
process(rss_url, SOURCES_indo)

Checking for updates...
Task complete. Sent 0 relevant articles.


# Malaysia

In [68]:
# SOURCES_malay = [
#     "newswav.com",
#     "thestar.com",
#     "malaymail.com",
#     "freemalaysiatoday.com",
#     "nst.com",
#     "bernama.com",
#     "thesun.my"
# ]

# full_query = query(SOURCES_malay)

# rss_url = f"https://news.google.com/rss/search?q={full_query}&hl=en-MY&gl=MY&ceid=MY:en"
# process(rss_url, SOURCES_malay)

# Bangladesh

In [69]:
SOURCES_bangladesh = [
    "thedailystar.net",
    "dhakatribune.com",
    "bdnews24.com",
    "livemint.com"
]

full_query = query(SOURCES_bangladesh)

rss_url = f"https://news.google.com/rss/search?q={full_query}&hl=en-BD&gl=BD&ceid=BD:en"
process(rss_url, SOURCES_bangladesh)

Checking for updates...
Task complete. Sent 0 relevant articles.


# Singapore

In [70]:
SOURCES_singapore = [
    "todayonline.com",
    "channelnewsasia.com",
    "sg.news.yahoo.com",
    "asiaone.com"
]

full_query = query(SOURCES_singapore)

rss_url = f"https://news.google.com/rss/search?q={full_query}&hl=en-SG&gl=SG&ceid=SG:en"
process(rss_url, SOURCES_singapore)

Checking for updates...
Task complete. Sent 0 relevant articles.


# Philippine

In [71]:
SOURCES_philippines = [
    "inquirer.net",
    "news.abs-cbn.com",
    "philstar.com",
    "rappler.com",
    "manilatimes.net",
    "manilatimes.net"
]

full_query = query(SOURCES_philippines)

rss_url = f"https://news.google.com/rss/search?q={full_query}&hl=en-PH&gl=PH&ceid=PH:en"
process(rss_url, SOURCES_philippines)

Checking for updates...
Task complete. Sent 0 relevant articles.


# Brunei

In [72]:
SOURCES_brunei = [
    "thebruneian.news",
    "borneobulletin.com.bn"
]

full_query = query(SOURCES_brunei)

rss_url = f"https://news.google.com/rss/search?q={full_query}&hl=en-BN&gl=BN&ceid=BN:en"
process(rss_url, SOURCES_brunei)

Checking for updates...
Task complete. Sent 0 relevant articles.
